In [70]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf
from typing import Optional
import requests
from datetime import datetime, timedelta
from abc import ABC, abstractmethod
import logging
from typing import List, Dict, Optional, Union
from dataclasses import dataclass, field


In [ ]:
@dataclass
class Asset(ABC):
    symbol: str
    _current_price: Optional[float] = None

    @property
    @abstractmethod
    def current_price(self) -> float:
        pass

@dataclass
class Portfolio:
    name: str
    positions: List[Position] = field(default_factory=list)

    def add_position(self, position: Position):
        if position.quantity <= 0:
            raise ValueError("quantity deve ser positiva")
        self.positions.append(position)

    def del_position(self, position: Position):
        if position not in self.positions:
            raise ValueError("Position not found")
        self.positions.remove(position)

    def total_value(self) -> float:
        return sum(position.total_value() for position in self.positions)

    def total_pnl(self) -> float:
        return sum(position.calculate_pnl() for position in self.positions)

    def __getitem__(self, index: int) -> Position:
        return self.positions[index]

@dataclass
class Position(ABC):
    asset: Asset
    quantity: float = 1

    @abstractmethod
    def calculate_pnl(self) -> float:
        pass

    def total_value(self) -> float:
        return self.asset.current_price * self.quantity

    def __repr__(self) -> str:
        entry = getattr(self, "entry_price", None)
        return f"Position({self.asset.symbol}, qty={self.quantity}, entry={entry})"

@dataclass
class LongPosition(Position):

    entry_price: float = None
    def __post_init__(self):
        if self.entry_price is None:
            self.entry_price = self.asset.current_price

    def calculate_pnl(self) -> float:
        return (self.asset.current_price - self.entry_price) * self.quantity

@dataclass
class ShortPosition(Position):

    entry_price: float = None

    def __post_init__(self):
        if self.entry_price is None:
            self.entry_price = self.asset.current_price

    def calculate_pnl(self) -> float:
        return (self.entry_price - self.asset.current_price) * self.quantity


#implementar futuramente
'''
@dataclass
  class Signal:
      symbol: str
      signal_type: str  # 'buy' ou 'sell'
      confidence: float
      timestamp: datetime

  class Strategy:
      def __init__(self, name: str):
          self.name: str = name
          self.signals: List[Signal] = []

      def generate_signals(self, data: pd.DataFrame) -> List[Signal]:
          """Retorna lista de sinais gerados"""
          pass'''

class PriceProvider(ABC):
    @abstractmethod
    def get_price(self, symbol: str) -> float:
        pass

@dataclass
class YahooFinanceProvider(PriceProvider):

    period: str = "1d"

    def get_price(self, symbol: str) -> float:
        data = yf.download(symbol, period=self.period)
        if data.empty:
            raise ValueError(f"Failed to fetch data for {symbol}")
        close = data['Close']
        if hasattr(close, 'columns'):
            close = close.iloc[:, 0]
        price = close.iloc[-1]
        return float(price)

@dataclass
class Crypto(Asset):

    provider: PriceProvider = None

    def __post_init__(self):
        if self.provider is None:
            self.provider = YahooFinanceProvider()

    @property
    def current_price(self) -> float:
        if self._current_price is None:
            logging.info(f"Preço não definido para {self.symbol}, buscando...")
            self._current_price = self.provider.get_price(self.symbol)
        return self._current_price

    def refresh_price(self):
        self._current_price = None

In [88]:
btc_price_1 = Crypto("BTC-USD")
eth_price_1 = Crypto("ETH-USD")

In [87]:
#eth_price_1.__repr__()
btc_price_1.__str__()

"Crypto(symbol='BTC-USD', _current_price=None, provider=YahooFinanceProvider(period='1d'))"

In [74]:
btc_price_1.refresh_price()
eth_price_1.refresh_price()

In [75]:
position_btc_1 = LongPosition(btc_price_1, entry_price=50000, quantity=0.3)
position_eth_1 = LongPosition(eth_price_1, quantity=2.5)


[*********************100%***********************]  1 of 1 completed


In [76]:
position_btc_1.__str__()

"LongPosition(asset=Crypto(symbol='BTC-USD', _current_price=None, provider=YahooFinanceProvider(period='1d')), quantity=0.3, entry_price=50000)"

In [77]:
portifolio_cripto = Portfolio("Crytpto")

In [78]:
portifolio_cripto.add_position(position_btc_1)
portifolio_cripto.add_position(position_eth_1)

In [79]:
portifolio_cripto.__str__()

"Portfolio(name='Crytpto', positions=[LongPosition(asset=Crypto(symbol='BTC-USD', _current_price=None, provider=YahooFinanceProvider(period='1d')), quantity=0.3, entry_price=50000), LongPosition(asset=Crypto(symbol='ETH-USD', _current_price=2253.867919921875, provider=YahooFinanceProvider(period='1d')), quantity=2.5, entry_price=2253.867919921875)])"

In [80]:
portifolio_cripto.total_value()

[*********************100%***********************]  1 of 1 completed


27376.384643554688

In [81]:
portifolio_cripto.total_pnl()

6741.71484375

In [82]:
for c in portifolio_cripto.positions:
    print(c.total_value())

21741.71484375
5634.6697998046875
